# Neo4j Aura Agent + LangChain

An **Aura Agent** is a managed agent that Neo4j hosts alongside your database. You configure it in
the Aura Console - its ontology, its tools, its instructions - and Neo4j runs it. It does its own
chain-of-thought reasoning over the graph and returns an answer.

Exposing it as an **MCP server** makes it callable from anywhere. In this notebook we'll connect one
to LangChain, where it becomes a tool in a larger agent: the Aura Agent handles graph reasoning, and
LangChain handles orchestration, conversation state, and everything else your application needs.

Nothing runs locally - the agent already runs in Aura.

In [2]:
!pip install --quiet --upgrade langchain langchain-openai langchain-mcp-adapters requests


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import json
import os
import time
from getpass import getpass

import requests

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

In [4]:
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key")

model = ChatOpenAI(model="gpt-5.4-mini")

## About the agent used here

This notebook connects to an **Investment Support Agent** built in the Neo4j Aura Console. It is
just an example - the integration below works with any Aura Agent, so use your own if you have one.

To create one:

1. In the [Aura Console](https://console.neo4j.io/), enable **GenAI assistance** for your
   organization and **Tool authentication** for your database.
2. Create an agent, point it at your instance, and configure its ontology, tools, and instructions.
3. Open **Configure → External access**, enable the **MCP server**, and update the agent.
4. Copy the MCP endpoint from the agent's `[…]` menu.

The [getting started tutorial](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/)
walks through it end to end.

Because all of the graph reasoning lives in the agent's own configuration, nothing below is specific
to this agent - only the questions in the demo cells assume investment data. Swap those for whatever
your agent knows about.

## Configuration

Two credentials and one URL, all from the Aura Console.

**Client credentials.** Profile menu → **Account settings** → **Client credentials** tab →
**Aura Agent & MCP** → **Create client credential**. Save the generated values, they're shown once.

> These are *not* Aura API keys. API keys authenticate against `api.neo4j.io` and work with the
> agent's REST endpoint; the MCP endpoint uses client credentials from the Aura Agent & MCP tab and
> a different token endpoint.

**MCP endpoint URL**, copied from the agent's `[…]` menu:

```
https://mcp.neo4j.io/agent?project_id=<PROJECT_ID>&agent_id=<AGENT_ID>
```

Set them as environment variables so nothing sensitive lands in the notebook:

```bash
export AURA_MCP_CLIENT_ID=...
export AURA_MCP_CLIENT_SECRET=...
export AURA_AGENT_MCP_URL="https://mcp.neo4j.io/agent?project_id=...&agent_id=..."
```

> An externally accessible agent incurs charges per [Neo4j Aura pricing](https://neo4j.com/pricing/).
> Availability can be switched off again from the same menu.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

AURA_MCP_TOKEN_URL = "https://mcp.neo4j.io/oauth/token"
AURA_MCP_AUDIENCE = "https://agent-mcp.neo4j.io"

CLIENT_ID = os.environ.get("AURA_MCP_CLIENT_ID") or getpass("Aura MCP client ID: ")
CLIENT_SECRET = os.environ.get("AURA_MCP_CLIENT_SECRET") or getpass("Aura MCP client secret: ")
AGENT_MCP_URL = os.environ.get("AURA_AGENT_MCP_URL") or input("Agent MCP endpoint URL: ").strip()

# Show the endpoint without leaking the IDs into notebook output.
print("Endpoint:", AGENT_MCP_URL.split("?")[0], "(project and agent IDs hidden)")

Endpoint: https://mcp.neo4j.io/agent (project and agent IDs hidden)


## Get a token

The agent MCP server supports two authorization flows: **user (authorization code)**, which is the
browser login an MCP client like Claude Desktop performs, and **machine-to-machine (client
credentials)**, for scripts and services. We use the second - no browser, so this runs from a
backend or a scheduled job.

Post a `client_credentials` grant to the gateway token endpoint with the fixed audience
`https://agent-mcp.neo4j.io`, then send the returned JWT as a bearer token.

> **The token endpoint allows 15 requests per hour per client ID.** Cache the token for its full
> `expires_in` window rather than fetching one per request. The helper below does that, so
> re-running cells reuses the cached token instead of spending quota.

In [7]:
_token = {"value": None, "expires_at": 0}


def get_token(force: bool = False) -> str:
    """Return a cached bearer token, fetching a new one only when it is near expiry."""
    if not force and _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    response = requests.post(
        AURA_MCP_TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AURA_MCP_AUDIENCE,
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    _token["value"] = payload["access_token"]
    _token["expires_at"] = time.time() + payload.get("expires_in", 3600)
    return _token["value"]


get_token()
print(f"Token cached, valid for {int(_token['expires_at'] - time.time())}s")

Token cached, valid for 86399s


## Check the endpoint

Call the MCP endpoint directly before handing it to a model. If the credentials or URL are wrong,
the error is obvious here; inside an agent transcript it isn't.

This also shows which tools the agent exposes, which is what the model will see.

In [9]:
response = requests.post(
    AGENT_MCP_URL,
    headers={
        "Authorization": f"Bearer {get_token()}",
        "Content-Type": "application/json",
        # Streamable HTTP can answer as JSON or as a server-sent event stream.
        "Accept": "application/json, text/event-stream",
    },
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    timeout=60,
)
response.raise_for_status()

body = response.text
if "text/event-stream" in response.headers.get("Content-Type", ""):
    body = next(line[5:] for line in body.splitlines() if line.startswith("data:"))

for t in json.loads(body)["result"]["tools"]:
    print(f"  {t['name']}")
    print(f"    {t.get('description', '').splitlines()[0][:200]}")

  Investment_Support_Agent
    Assists with exploring companies, their partnerships, key personnel, and relevant articles for investment purposes.


## Connect it to LangChain

`MultiServerMCPClient` from `langchain-mcp-adapters` connects to the remote endpoint and returns its
tools as LangChain tools. The bearer token goes in the headers.

Note what the system prompt does *not* contain: no schema, no Cypher guidance, no description of the
graph. All of that lives in the Aura Agent's configuration. The LangChain agent's job is to decide
*when* to consult it and how to present the answer.

In [10]:
def mcp_client():
    """Connect to the agent's MCP endpoint with a current token.

    MCP tools capture their headers when the client is created, so call this again after the
    token expires. `get_token()` returns the cached value until it is close to expiring.
    """
    return MultiServerMCPClient({
        "aura-agent": {
            "transport": "streamable_http",
            "url": AGENT_MCP_URL,
            "headers": {"Authorization": f"Bearer {get_token()}"},
        }
    })


client = mcp_client()
aura_tools = await client.get_tools()

print("Tools from Aura:", [t.name for t in aura_tools])


Tools from Aura: ['Investment_Support_Agent']


In [11]:
system_prompt = """
You help users with investment research questions.

A hosted Neo4j Aura Agent has access to the investment knowledge graph - use its tools for anything
about companies, holdings, or relationships in that data. Pass the user's question through clearly
rather than rewriting it into keywords.

Present what comes back in your own words, and say plainly if the agent could not answer rather than
filling the gap from your own knowledge.
"""

agent = create_agent(model, aura_tools, system_prompt=system_prompt)

Let's test it!

In [12]:
prompt = "Who are the competitors of OpenAI in the AI industry?"

async for event in agent.astream(
    {"messages": [{"role": "user", "content": prompt}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who are the competitors of OpenAI in the AI industry?
================================== Ai Message ==================================
Tool Calls:
  Investment_Support_Agent (call_pBpLTYn7kBayUbm6pAQvUwap)
 Call ID: call_pBpLTYn7kBayUbm6pAQvUwap
  Args:
    query: Who are the competitors of OpenAI in the AI industry?
================================= Tool Message =================================
Name: Investment_Support_Agent

[{'type': 'text', 'text': 'The competitors of OpenAI in the AI industry include:\n\n*   Google\n*   Meta Platforms\n*   Alphabet\n*   Cohere\n*   Amazon Web Services (AWS)\n*   ChatGPT (Note: This is often considered a product of OpenAI, but listed in the competitive landscape)\n*   MedicalGPT\n*   ThreatGPT\n*   DeepMind\n*   Alibaba\n*   Amazon\n*   Uber Technologies\n*   Midjourney\n*   YouTube\n*   Bard', 'id': 'lc_51e710d2-6819-4787-bfdc-19ac0ae3c432'}]
=======================

The tool call in the trace goes out to Aura, where the hosted agent plans, runs its own queries, and
reasons over the results. What returns is an answer, not rows - the LangChain agent then decides how
to present it.

That division is the point of this integration. The graph logic lives with the graph and is
maintained in the Console; the conversational layer lives in your application. Changing the agent's
ontology or tools takes effect without touching this notebook.

## Summary

In this notebook, we connected a hosted Neo4j Aura Agent to LangChain:

1. **Machine-to-machine authentication** - client credentials from the Aura Agent & MCP tab exchanged for a bearer token at the gateway endpoint, with no browser step
2. **Token caching** - the endpoint allows 15 requests per hour per client ID, so each token is held for its full lifetime
3. **MCP connection** - `MultiServerMCPClient` turns the remote agent's tools into LangChain tools

The Aura Agent is a hosted *agent*, not a database connection: it reasons over the graph using the
ontology you configured in the Console and returns an answer rather than rows. Because that
configuration lives in Aura, updating the agent's behaviour doesn't require a code change here.

> **Token expiry.** MCP tools capture their headers when the client is created, so the token doesn't
> refresh itself. Rebuild the client when it expires, or front the endpoint with a proxy that injects
> a current token per request.

### Resources

- [Aura Agent documentation](https://neo4j.com/docs/aura/aura-agent/) - configuration, external access, both authorization flows
- [Aura Agent getting started](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/) - building an agent from scratch